# Location Data Visualization
This notebook displays locations from our FastAPI service using Folium.

In [1]:
import folium
import pandas as pd
import requests
import webbrowser
from folium.plugins import MarkerCluster
import osmnx as ox
import time
from IPython.display import display, clear_output

def refresh_map():
    try:
        print("Fetching location data...")
        # Fetch data from FastAPI
        response = requests.get('http://localhost:8000/locations/')
        locations = response.json()
        
        # Create DataFrame
        df = pd.DataFrame(locations)
        
        print("Creating base map...")
        if not df.empty:
            center = [df.loc[df['id'].idxmin(), 'latitude'], df.loc[df['id'].idxmin(), 'longitude']]
        else:
            center = [40.7128, -74.0060]  # NYC coordinates as default
        
        # Create map
        m = folium.Map(location=center, zoom_start=15)
        marker_cluster = MarkerCluster()
        
        # Add markers
        for _, row in df.iterrows():
            folium.Marker(
                location=[row['latitude'], row['longitude']],
                popup=f"<b>{row['name']}</b><br>{row['description']}",
                tooltip=row['name']
            ).add_to(marker_cluster)
        
        # If we have points, add road network
        if len(df) >= 4:
            print("Calculating road network area...")
            # Get the bounding box of our points
            min_lat = df['latitude'].min()
            max_lat = df['latitude'].max()
            min_lon = df['longitude'].min()
            max_lon = df['longitude'].max()
            
            # Reduce the area size for faster loading
            padding = 0.0005  # About 50m instead of 100m
            north = max_lat + padding
            south = min_lat - padding
            east = max_lon + padding
            west = min_lon - padding
            
            # Get road network with timeout and simplified settings
            try:
                print("Fetching road network (this may take a moment)...")
                # Configure OSMNX for faster processing
                ox.settings.use_cache = True
                ox.settings.log_console = False
                ox.settings.timeout = 30
                
                # Get simplified road network using average center point
                G = ox.graph_from_point(
                    center,
                    dist=300,  # Reduced to 300m radius for faster loading
                    network_type='drive',
                    simplify=True
                )
                
                print("Processing road network...")
                # Convert to GeoJSON and add to map
                gdf_edges = ox.graph_to_gdfs(G, nodes=False)
                folium.GeoJson(
                    gdf_edges,
                    style_function=lambda x: {
                        'color': '#FF0000',
                        'weight': 2,
                        'opacity': 0.7
                    }
                ).add_to(m)
            except Exception as e:
                print(f"Warning: Could not fetch road network: {e}")
            
            print("Adding square outline...")
            # Add square outline
            square_coords = df.sort_values('id').iloc[-4:][['latitude', 'longitude']].values
            folium.PolyLine(
                locations=square_coords.tolist() + [square_coords[0].tolist()],
                color='blue',
                weight=3,
                opacity=0.8,
                popup='Generated Square'
            ).add_to(m)
        
        marker_cluster.add_to(m)
        
        print("Saving and opening map...")
        # Save and open map
        output_file = 'map.html'
        m.save(output_file)
        webbrowser.open(output_file, new=2)
        print("Map generation complete!")
        
    except Exception as e:
        print(f"Error generating map: {e}")

# Display the map
refresh_map()

Fetching location data...
Creating base map...
Calculating road network area...
Fetching road network (this may take a moment)...
Processing road network...
Adding square outline...
Saving and opening map...
Map generation complete!
